In [0]:
%fs ls dbfs:/FileStore/data/schema

In [0]:
DROP DATABASE IF EXISTS demodb CASCADE;
CREATE DATABASE demodb;
USE demodb;

In [0]:
CREATE OR REPLACE TABLE people (
  id INT,
  firstName STRING,
  lastName STRING
) USING DELTA;

#####Schema Validations Summary
- `INSERT`
  - Column matching by position, New columns not allowed
- `OVERWRITE`
  - Column matching by position, New columns not allowed
- `MERGE .. INSERT`
  - Column matching by name, New columns ignored
- `DataFrame Append`
  - Column matching by name, New columns not allowed
- `Data Type Mismatch`
  - Not allowed in any case

##### INSERT INTO

- Column matching by position
- New columns not allowed

In [0]:
INSERT INTO people
SELECT id, fname, lname
FROM JSON.`/Volumes/workspace/default/data/schema/people.json`

In [0]:
SELECT * FROM people

In [0]:
INSERT INTO people
SELECT id, fname, lname, dob 
FROM json.`/Volumes/workspace/default/data/schema/people.json`

##### INSERT OVERWRITE
- Column matching by position
- New columns not allowed

In [0]:
INSERT OVERWRITE people
SELECT id, fname, lname, dob
FROM json.`/Volumes/workspace/default/data/schema/people.json`

##### MERGE .. INSERT

In [0]:
SELECT id, fname, lname FROM json.`/Volumes/workspace/default/data/schema/people_2.json`

In [0]:
MERGE INTO people T
USING (SELECT id, fname, lname FROM json.`/Volumes/workspace/default/data/schema/people_2.json`) S
ON T.id = S.id
WHEN NOT MATCHED THEN INSERT *  

In [0]:
MERGE INTO people T
USING 
( SELECT id, fname firstName, lname lastName
  FROM json.`/Volumes/workspace/default/data/schema/people_2.json`
) S
ON T.id = S.id
WHEN NOT MATCHED THEN INSERT *  

In [0]:
SELECT * FROM people

In [0]:
SELECT id, fname firstName, lname lastName, dob 
FROM json.`/Volumes/workspace/default/data/schema/people_3.json`

In [0]:
MERGE INTO people T
USING 
(   SELECT id, fname firstName, lname lastName, dob 
    FROM json.`/Volumes/workspace/default/data/schema/people_3.json`
) S
ON T.id = S.id
WHEN NOT MATCHED THEN INSERT *

In [0]:
select * from people

##### Dataframe append

- Column matching by position not allowed
- Column matching by name is allowed


In [0]:
SELECT * FROM people

In [0]:
%python
people_schema = "id INT, fname STRING, lname STRING"
people_df =  spark.read.schema(people_schema).json("/Volumes/workspace/default/data/schema/people_2.json")
people_df.write.format("delta").mode("append").saveAsTable("people")

In [0]:
%python
people_schema = "id INT, fname STRING, lname STRING"

people_df =  (
  spark
    .read
    .schema(people_schema)
    .json("/Volumes/workspace/default/data/schema/people_2.json")
    .withColumnRenamed("fname", "firstName")
    .withColumnRenamed("lname", "lastName")
)

people_df.write.format("delta").mode("append").saveAsTable("people")

In [0]:
SELECT * FROM people

In [0]:
%python
people_schema = "id INT, fname STRING, lname STRING, dob DATE"

people_df =  (
  spark
    .read
    .schema(people_schema)
    .json("/Volumes/workspace/default/data/schema/people_2.json")
    .withColumnRenamed("fname", "firstName")
    .withColumnRenamed("lname", "lastName")
)

display(people_df)

In [0]:
%python
people_df.write.format("delta").mode("append").saveAsTable("people")

##### Cleanup

In [0]:
DROP DATABASE IF EXISTS demodb CASCADE